## Test Grid SANOS @ torch



In [7]:
%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.widb
%aimport quantdata.widb.client
%aimport quantdata.widb.day
%aimport quantdata.widb.timeseries
%aimport quantdata.dysanos
%aimport quantdata.utils.backbone
%aimport quantdata.torch
%aimport quantdata.torch.bbtorch
%aimport quantdata.torch.ffw
%aimport quantdata.dysanos.simple
%aimport quantdata.dysanos.simple.sampler
%aimport quantdata.dysanos.simple.model
%aimport quantdata.sanos.grid
%aimport quantdata.sanos.grid.fitter

from quantdata.dysanos.simple.aec import load_data
data = load_data(["SPX"]).data['SPX']
print(f"/done; columns {data.keys()}")



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
00: Generating samples for 1 tickers SPX with from 2015-01-01 to 2025-07-01... 

00: Generating samples for 1 tickers SPX with from 2015-01-01 to 2025-07-01... done; this took 2s.
00: Found the following number of samples per ticker: SPX: 2617.
00: Found date range 2015-01-02 to 2025-07-01 with 2617 total dates. First validation date for 25 steps per sample is 2023-05-01.
00: Using up to 1000 options per date.
01:   Ticker SPX: using 411 training samples (accross 2079 training dates), and 103 validation samples (accross 538 validation dates).
00: Created 411 training samples and 103 validation samples based on the date split at 2023-05-01. That implies 20% of all samples are validation samples.
00: Total time taken: 2s.
/done; columns dict_keys(['dates', 'ask_pure_delta', 'ask_pure_dk', 'ask_pure_gamma', 'ask_pure_price', 'ask_pure_theta', 'ask_pure_vega', 'ask_vol', 'bid_pure_delta', 'bid_pure_dk', 'bid_pure_gamma', 'bid_pure_price', 'bid_pure_theta', 'bid_pure_vega', 'bid_vol', 'call_ask', 'call_bid', 'df', 'fwd', 'is_am', 'is_american', 'is_valid', 'log_open_int

In [16]:
from quantdata import env, SubDir



num_samples = len(data.dates)
slc_samples = slice(num_samples-5,num_samples-1) # 3 samples



sd = SubDir("/mnt/hans/python/quant_data/tests/stored_data")

r = dict(
    strikes     = data.pure_strike[slc_samples,:],
    is_valid    = data.is_valid[slc_samples,:],
    tte_y       = data.tte_y[slc_samples,:],
    bid         = data.bid_pure_price[slc_samples,:],
    ask         = data.ask_pure_price[slc_samples,:],
    tte         = data.tte_y[slc_samples,:],
    bid_vol     = data.bid_vol[slc_samples,:],
    ask_vol     = data.ask_vol[slc_samples,:]
    )

env._test_data_locations.training.write("grid_sanos_four_samples", r, fmt=SubDir.PYTREE_HDF5, raise_on_error=True )


True

In [ ]:
%aimport quantdata.sanos.grid
%aimport quantdata.sanos.grid.fitter

from cdxcore.bs import bs
from cdxcore.dynaplot import figure, store
from cdxcore.verbose import Context
from cdxcore.pretty import PrettyObject
from quantdata.biz_calendar import DAYS_PER_YEAR
from quantdata.sanos.grid import GridGeometry, GridGeometryGenerator, GridSanos
from quantdata.sanos.fitter import Eps, Fitter
import numpy as np

from quantdata.torch.bbtorch import BackboneTorch
from quantdata.utils.backbone import BackboneNumpy

def fit(data):
    # common numpy
    # -------------

    grid        = GridGeometryGenerator(
                            horizon_t  = 1., 
                            num_times  = 10,
                            nms_range  = (-4,4),
                            num_nms    = 60,
                            dlv_min    = 0.01,
                            dlv_max    = 5.,
                            fwdvol_min = 0.0,
                            vol_interp = "lin_var",
                            lower_scale = 0.2,
                            upper_scale = 1.5
            ).create(dtype=np.float64)

    eps         = Eps()
    eps.set_all( 1E-3 )
    fitter      = Fitter( geometry = grid, eps = eps, debug=True, max_error = None, mid_fit_weight = 1. )
    bbnp        = BackboneNumpy()
    bbtr        = BackboneTorch("cpu")
    np_sanos    = GridSanos( grid, bb = bbnp )
    tr_sanos    = GridSanos( grid, bb = bbtr )

    # sample
    # -----

    num_samples = len(data.dates)
    slc_samples = slice(num_samples-5,num_samples-1) # 3 samples

    np_sample = PrettyObject(
        strikes     = data.pure_strike[slc_samples,:],
        is_valid    = data.is_valid[slc_samples,:],
        tte_y       = data.tte_y[slc_samples,:],
        bid         = data.bid_pure_price[slc_samples,:],
        ask         = data.ask_pure_price[slc_samples,:],
        tte         = data.tte_y[slc_samples,:],
        bid_vol     = data.bid_vol[slc_samples,:],
        ask_vol     = data.ask_vol[slc_samples,:]
        )
    
    np_opt_interp  = grid.time_interpolation_weights( np_sample.tte_y )
 
    # Fit
    # ---
    # Works in NP only, and not in grid mode

    np_sample.fwd_vols = np.full( (grid.T,), 0.05, dtype=np.float64 )

    T, N  = grid.TN
    M     = np_sample.strikes.shape[-1] 
    np_sample.dlvsqs = np.zeros((np_sample.strikes.shape[0],T,N-2))
    np_sample.fit_prices = np.zeros((np_sample.strikes.shape[0],M))
    np_sample.densities = np.zeros((np_sample.strikes.shape[0],T,N))

    for isample in range(np_sample.strikes.shape[0]):
        np_sample.dlvsqs[isample,:], \
        np_sample.fit_prices[isample,:], \
        np_sample.densities[isample,:] = \
            fitter( fwd_vols     = np_sample.fwd_vols,
                    opt_strikes  = np_sample.strikes[isample,:],
                    opt_bid      = np_sample.bid[isample,:],
                    opt_ask      = np_sample.ask[isample,:],
                    opt_tte      = np_sample.tte[isample,:],
                    opt_is_calls = None,
                    opt_mask     = np_sample.is_valid[isample,:],
                    return_only_dlvsqs = False,
                    verbose      = Context.all ) 
    np_sample.fit_vols    = bs.implied(np_sample.strikes, np_sample.fit_prices, sqrtT = np.sqrt(np_sample.tte_y) )
    tr_sample             = PrettyObject({ k: bbtr.to_array(v) for k,v in np_sample.items() })
    tr_opt_interp         = { k: bbtr.to_array(v) for k,v in np_opt_interp.items() }

    # Reprice (numpy)
    # ---------------

    np_sanos     = GridSanos( grid, bb = bbnp )
    np_sample.test_density = np_sanos.densities( np_sample.dlvsqs )
    np_sample.test_prices  = np_sanos.price(
                        np_sample.fwd_vols[None,:],
                        np_sample.test_density,
                        opt_strikes = np_sample.strikes,
                        opt_tte     = np_sample.tte,
                        opt_mask    = np_sample.is_valid,
                        opt_is_call = None,
                        opt_interp  = np_opt_interp
    )
    np_sample.test_vols    = bs.implied(np_sample.strikes, np_sample.test_prices, sqrtT = np.sqrt(np_sample.tte_y) )

    # Torch
    # -----

    tr_sanos     = GridSanos( grid, bb = bbtr )
    tr_sample.test_density = tr_sanos.densities( tr_sample.dlvsqs )
    tr_sample.test_prices  = tr_sanos.price(
                        tr_sample.fwd_vols[None,:],
                        tr_sample.test_density,
                        opt_strikes = tr_sample.strikes,
                        opt_tte     = tr_sample.tte,
                        opt_mask    = tr_sample.is_valid,
                        opt_is_call = None,
                        opt_interp  = tr_opt_interp
    )
    tr_sample.test_vols    = bbtr.to_array(bs.implied(
                        tr_sample.strikes.cpu().numpy(),
                        tr_sample.test_prices.cpu().numpy(),
                        sqrtT = np.sqrt(tr_sample.tte_y.cpu().numpy())
    ))

    # validate
    # --------

    assert set(np_sample) == set(tr_sample), f"Expected same keys in numpy and torch samples, found {set(np_sample)} and {set(tr_sample)}"
    for k in np_sample:
        v_np = np_sample[k]
        v_tr = tr_sample[k].cpu().numpy()
        assert np.allclose( v_np, v_tr, atol=1e-6 ), f"Expected numpy and torch values to match for key '{k}', max abs diff {np.max(np.abs(v_np - v_tr))}"
    print("Everything worked!")

fit(data)


00: Solving for implied DLVs (squared) for 1000 options. Mean half spread is 5.2755e-05... 

/mnt/c/Users/hans/OneDrive/Python3/quant_data/quantdata/sanos/grid/fitter.py:192: UserWarning: Found option with expiry 1.484 beyond the last model expiry 1. While functionaly feasible, this is likely to cause fitting issues. Expand expiry range.


00: Solving for implied DLVs (squared) for 1000 options. Mean half spread is 5.2755e-05... done with relative excess error conditional mean 2.1776e+00, std 5.3020e+00, 95th percentile 8.1844e+00, 99th percentile 3.3886e+01, 99.9th percentile 4.3259e+01 and max 4.3540e+01. This took 0.22s.
00: Computing DLV squared's... done; this tool 0.788ms.
00: Solving for implied DLVs (squared) for 1000 options. Mean half spread is 5.5631e-05... 

/mnt/c/Users/hans/OneDrive/Python3/quant_data/quantdata/sanos/grid/fitter.py:192: UserWarning: Found option with expiry 2.472 beyond the last model expiry 1. While functionaly feasible, this is likely to cause fitting issues. Expand expiry range.


00: Solving for implied DLVs (squared) for 1000 options. Mean half spread is 5.5631e-05... done with relative excess error conditional mean 3.3390e+00, std 8.9235e+00, 95th percentile 1.8907e+01, 99th percentile 4.8873e+01, 99.9th percentile 5.4121e+01 and max 5.8501e+01. This took 0.24s.
00: Computing DLV squared's... done; this tool 0.95ms.
00: Solving for implied DLVs (squared) for 1000 options. Mean half spread is 1.5826e-04... 

/mnt/c/Users/hans/OneDrive/Python3/quant_data/quantdata/sanos/grid/fitter.py:192: UserWarning: Found option with expiry 2.468 beyond the last model expiry 1. While functionaly feasible, this is likely to cause fitting issues. Expand expiry range.


00: Solving for implied DLVs (squared) for 1000 options. Mean half spread is 1.5826e-04... done with relative excess error conditional mean 3.9795e+00, std 7.0645e+00, 95th percentile 1.8538e+01, 99th percentile 3.2836e+01, 99.9th percentile 3.8597e+01 and max 3.8958e+01. This took 0.28s.
00: Computing DLV squared's... done; this tool 0.765ms.
00: Solving for implied DLVs (squared) for 1000 options. Mean half spread is 7.2257e-05... 

/mnt/c/Users/hans/OneDrive/Python3/quant_data/quantdata/sanos/grid/fitter.py:192: UserWarning: Found option with expiry 2.464 beyond the last model expiry 1. While functionaly feasible, this is likely to cause fitting issues. Expand expiry range.


00: Solving for implied DLVs (squared) for 1000 options. Mean half spread is 7.2257e-05... done with relative excess error conditional mean 2.2170e+00, std 6.1303e+00, 95th percentile 9.3005e+00, 99th percentile 3.5080e+01, 99.9th percentile 3.6995e+01 and max 3.7461e+01. This took 0.27s.
00: Computing DLV squared's... done; this tool 0.77ms.
spots (4, 1000, 40)
opt_strikes[...,:,None] (4, 1000, 1)
opt_sqrtVars[...,:,None] (4, 1000, 1)
opt_mask[...,:,None] (4, 1000, 1)
opt_dweight[...,:,None] (4, 1000, 1)
calls_prev (4, 1000, 40)
spots torch.Size([4, 1000, 40])
opt_strikes[...,:,None] torch.Size([4, 1000, 1])
opt_sqrtVars[...,:,None] torch.Size([4, 1000, 1])
opt_mask[...,:,None] torch.Size([4, 1000, 1])
opt_dweight[...,:,None] torch.Size([4, 1000, 1])
calls_prev torch.Size([4, 1000, 40])
Everything worked!


/mnt/c/Users/hans/OneDrive/Python3/quant_data/quantdata/torch/bbtorch.py:42: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
